In [1]:
!pip install datasets transformers torchaudio evaluate


Active code page: 1252
Defaulting to user installation because normal site-packages is not writeable


In [13]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset
import librosa
from transformers import WhisperProcessor


In [14]:
BASE_PATH = r"C:\Users\satya\Downloads\JoshTalks_ASR_Assignment\task\data"
TRAINING_CSV = os.path.join(BASE_PATH, "processed_training_data", "training_data_segmented.csv")

df = pd.read_csv(TRAINING_CSV)

print("Total Samples:", len(df))
df.head()


Total Samples: 5941


,audio_path,text
0,C:\Users\satya\Downloads\JoshTalks_ASR_Assignm...,अब काफी अच्छा होता है क्योंकि उनकी जनसंख्या बह...
1,C:\Users\satya\Downloads\JoshTalks_ASR_Assignm...,अनुभव करके कुछ लिखना था तो वह तो बिना देखिए नह...
2,C:\Users\satya\Downloads\JoshTalks_ASR_Assignm...,जंगल का सफर होता है जब हम रहने के लिए गए थे ना...
3,C:\Users\satya\Downloads\JoshTalks_ASR_Assignm...,पहली बारी था क्योंकि चलना नहीं आता न वहाँ का ज...
4,C:\Users\satya\Downloads\JoshTalks_ASR_Assignm...,हां तो फिर वहां जो दिन भर तो दिन में तो खोजने ...


In [15]:
processor = WhisperProcessor.from_pretrained(
    "openai/whisper-small",
    language="hi",
    task="transcribe"
)


In [16]:
class WhisperDataset(Dataset):
    def __init__(self, dataframe, processor):
        self.df = dataframe
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        audio_path = self.df.iloc[idx]["audio_path"]
        text = self.df.iloc[idx]["text"]

        audio, sr = librosa.load(audio_path, sr=16000)

        input_features = self.processor.feature_extractor(
            audio, sampling_rate=16000
        ).input_features[0]

        labels = self.processor.tokenizer(text).input_ids

        return {
            "input_features": torch.tensor(input_features),
            "labels": torch.tensor(labels)
        }


In [17]:
train_dataset = WhisperDataset(df, processor)

print("Dataset Ready ✅")
print("Length:", len(train_dataset))


Dataset Ready ✅
Length: 5941
